## Get API data into a variable

In [ ]:
import requests
import json
from os import getenv

api_key = getenv("CORESIGNAL_API_KEY")
api_endpoint = "https://api.coresignal.com/cdapi/v2/job_base/search/filter"

headers = {
    "apikey": api_key,
    "Content-Type": "application/json"
}
body = {
    "last_updated_gte": "2026-04-21 00:00:00",
    "title": "data engineer"
}

all_results = []
current_page = 1
next_page_after = None

while True:
    url = api_endpoint
    if next_page_after:
        url = f"{api_endpoint}?after={next_page_after}"

    response = requests.post(url, headers=headers, json=body)
    
    total_pages = response.headers.get('x-total-pages', 'unknown')
    total_results = response.headers.get('x-total-results', 'unknown')
    next_page_after = response.headers.get('x-next-page-after')
    
    print(f"Page {current_page}/{total_pages} | Total Results: {total_results}")
    
    try:
        page_results = response.json()
        all_results.extend(page_results)
        print(f"  Retrieved {len(page_results)} results from this page")
    except json.JSONDecodeError:
        print(f"  Error parsing JSON response: {response.text}")
        break
    
    if not next_page_after or current_page >= int(total_pages):
        break
    
    current_page += 1
    if current_page >= 5:
        break

print(f"\n--- Summary ---")
print(f"Total job IDs collected: {len(all_results)}")
print(f"Sample IDs: {all_results[:5] if all_results else 'No results'}")

In [3]:
ids_file_path = "/home/caroline/Downloads/all_results_200.csv"
id_file_endpoint = "https://api.coresignal.com/cdapi/v2/data_requests/job_base/id_file"
headers_ids = {
    "apikey": api_key,
    "accept": "application/json"
}

with open(ids_file_path, 'rb') as f:
    files = {
        'ids_file': (f.name, f, 'text/csv'),
    }
    data = {
        'limit': 200
    }
    response_ids = requests.post(id_file_endpoint, headers=headers_ids, files=files, data=data)
    print(f"Response Status Code: {response_ids.status_code}")
    print(f"Response: {response_ids.text}")

try:
    response_data = response_ids.json()
    request_id = response_data.get('request_id')
    print(f"\n✓ Request successful!")
    print(f"Request ID: {request_id}")
    print(f"Full Response: {json.dumps(response_data, indent=2)}")
except json.JSONDecodeError:
    print("Could not parse as JSON")

#2b07864e-dd67-4915-a955-a51b519ecabf

Response Status Code: 201
Response: {"request_id":"2b07864e-dd67-4915-a955-a51b519ecabf"}

✓ Request successful!
Request ID: 2b07864e-dd67-4915-a955-a51b519ecabf
Full Response: {
  "request_id": "2b07864e-dd67-4915-a955-a51b519ecabf"
}


In [12]:
## Retrieve bulk collect data using request ID
import gzip
import io

request_id = "2b07864e-dd67-4915-a955-a51b519ecabf"
bulk_data_endpoint = f"https://api.coresignal.com/cdapi/v2/data_requests/{request_id}/files"

headers_get = {
    "apikey": api_key,
    "accept": "application/json"
}

# Check status and retrieve files
response_get = requests.get(bulk_data_endpoint, headers=headers_get)
print(f"Response Status Code: {response_get.status_code}")

try:
    files_data = response_get.json()
    print(f"Files Response: {json.dumps(files_data, indent=2)}")
    
    # Extract file paths - they come as relative paths in data_request_files
    if 'data_request_files' in files_data:
        file_paths = files_data['data_request_files']
        print(f"\nFound {len(file_paths)} file(s)")
        
        # Download each file
        bulk_results = []
        for file_path in file_paths:
            # Construct download URL - use the full path with slash separators
            file_url = f"{bulk_data_endpoint}/{file_path}"
            print(f"Downloading from: {file_url}")
            
            file_response = requests.get(file_url, headers=headers_get, stream=True)
            print(f"  Response Status: {file_response.status_code}")
            print(f"  Content-Type: {file_response.headers.get('content-type')}")
            
            if file_response.status_code == 200:
                try:
                    # Decompress gzip if needed
                    if file_path.endswith('.gz'):
                        decompressed_content = gzip.decompress(file_response.content)
                        # File is newline-delimited JSON (NDJSON), parse line by line
                        text_content = decompressed_content.decode('utf-8')
                        for line in text_content.strip().split('\n'):
                            if line:
                                record = json.loads(line)
                                bulk_results.append(record)
                    else:
                        # Try parsing as JSON array first, then as NDJSON
                        try:
                            file_content = file_response.json()
                            if isinstance(file_content, list):
                                bulk_results.extend(file_content)
                            else:
                                bulk_results.append(file_content)
                        except json.JSONDecodeError:
                            # Parse as NDJSON
                            text_content = file_response.text
                            for line in text_content.strip().split('\n'):
                                if line:
                                    record = json.loads(line)
                                    bulk_results.append(record)
                    
                    print(f"  ✓ Retrieved {len(bulk_results)} records from file")
                except Exception as e:
                    print(f"  Error decompressing/parsing file: {e}")
                    import traceback
                    traceback.print_exc()
            else:
                print(f"  Error: HTTP {file_response.status_code}")
                print(f"  Response: {file_response.text[:200]}")
        
        print(f"\n✓ All data retrieved successfully!")
        print(f"Total records in bulk_results: {len(bulk_results)}")
        if bulk_results:
            print(f"Sample record keys: {list(bulk_results[0].keys())}")
            print(f"Sample record: {json.dumps(bulk_results[0], indent=2)[:500]}")
    else:
        print("No files found in response. Request may still be processing.")
        print(f"Check back later with request_id: {request_id}")
        
except json.JSONDecodeError as e:
    print(f"Error parsing response: {e}")
    print(f"Response text: {response_get.text}")

Response Status Code: 200
Files Response: {
  "data_request_files": [
    "json/part-00000-7cf25e50-7214-4751-ba48-0dd326900896-c000.json.gz"
  ]
}

Found 1 file(s)
  Response Status: 200
  Content-Type: binary/octet-stream
  ✓ Retrieved 200 records from file

✓ All data retrieved successfully!
Total records in bulk_results: 200
Sample record keys: ['id', 'created', 'last_updated', 'time_posted', 'title', 'description', 'seniority', 'employment_type', 'location', 'url', 'hash', 'company_id', 'company_name', 'external_url', 'company_url', 'deleted', 'application_active', 'salary', 'applicants_count', 'linkedin_job_id', 'country', 'redirected_url', 'job_company_website', 'job_industry_collection', 'job_functions_collection']
Sample record: {
  "id": 269545383,
  "created": "2024-09-04 06:20:17",
  "last_updated": "2026-04-22 12:28:37",
  "time_posted": "2 hours ago",
  "title": "Data Quality Assurance Engineer",
  "description": "Job Description\n\nAbout you\n\nYou are someone who wants 

## Save data to local database

In [20]:
# Save bulk_results to local PostgreSQL database table
import psycopg2
from datetime import datetime

database_user = getenv("DB_USER")
database_password = getenv("DB_PASSWORD")

conn = psycopg2.connect(
    host="localhost",
    database="market_fit",
    user=database_user,
    password=database_password
)

cur = conn.cursor()

# Insert job records into database
inserted_count = 0
error_count = 0

for result in bulk_results:
    try:
        # Parse timestamp fields
        created_at = None
        if result.get('created'):
            try:
                created_at = datetime.strptime(result.get('created'), "%Y-%m-%d %H:%M:%S")
            except:
                created_at = None
        
        updated_at = None
        if result.get('last_updated'):
            try:
                updated_at = datetime.strptime(result.get('last_updated'), "%Y-%m-%d %H:%M:%S")
            except:
                updated_at = None
        
        cur.execute("""
            INSERT INTO public.jobs 
            (source_id, id, url, application_active, deleted, created_at, updated_at, 
             title, description, seniority, employment_type, external_url, location, 
             country, applicants_count, company_id)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (id) DO NOTHING
        """, (
            1,  # source_id (hardcoded as 1, adjust as needed)
            result.get('id'),
            result.get('url'),
            1 if result.get('application_active') else 0,
            1 if result.get('deleted') else 0,
            created_at,
            updated_at,
            result.get('title'),
            result.get('description'),
            result.get('seniority'),
            result.get('employment_type'),
            result.get('external_url'),
            result.get('location'),
            result.get('country'),
            result.get('applicants_count'),
            result.get('company_id')
        ))
        inserted_count += 1
    except Exception as e:
        error_count += 1
        print(f"Error inserting record {result.get('id')}: {e}")

conn.commit()
cur.close()
conn.close()

print(f"✓ Database insertion complete!")
print(f"  Successfully inserted: {inserted_count}")
print(f"  Errors: {error_count}")

✓ Database insertion complete!
  Successfully inserted: 200
  Errors: 0
